# Seguimiento mensual de consignación SERMOTOR

Este notebook recibe uno o varios conteos por referencia y produce la tabla de
decisión mensual. El archivo mínimo contiene:
`referencia, unidades_en_bodega, fecha_conteo`.

Puede añadirse `reposiciones` como cuarta columna. Si se omite, se asume cero y la
salida lo informa explícitamente.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.analyze_sermotor_consignacion import ceil_even, normalize_designation

ARCHIVO_CONTEO = Path("outputs/sermotor_nivel_inicial/plantilla_conteo_ejemplo.csv")
MESES_COBERTURA = 2.5
CONTEOS_PARA_PROMEDIO = 3

# Niveles aprobados al iniciar el acuerdo. Quedan dentro del notebook para que
# este seguimiento solo requiera el CSV de conteo.
NIVELES_ACORDADOS = {'6314 | CAUCHO | C3': 4, '6312 | CAUCHO | C3': 6, '6311 | CAUCHO | C3': 8, '6313 | CAUCHO | C3': 4, '6313 | METAL | C3': 4, '6309 | CAUCHO | C3': 12, '6206 | METAL | C3': 38, '6310 | CAUCHO | C3': 6, '6309 | METAL | C3': 12, '6312 | METAL | C3': 4, '6308 | METAL | C3': 12, '6308 | CAUCHO | C3': 10, '6306 | METAL | C3': 20, '6208 | CAUCHO | C3': 14, '6306 | CAUCHO | C3': 16, '6206 | CAUCHO | C3': 24, '6208 | METAL | C3': 14, '6311 | METAL | C3': 4, '6205 | METAL | C3': 42, '6209 | CAUCHO | C3': 10, '6205 | CAUCHO | C3': 34, '6307 | METAL | C3': 10, '6207 | METAL | C3': 14, '6212 | CAUCHO | C3': 4, '6212 | METAL | C3': 4, '6210 | CAUCHO | C3': 6, '6211 | CAUCHO | C3': 4, '6204 | METAL | C3': 26, '6203 | CAUCHO | C3': 26, '6209 | METAL | C3': 6, '6204 | CAUCHO | C3': 20, '6210 | METAL | C3': 4, '6307 | CAUCHO | C3': 4, '6203 | METAL | C3': 22, '6305 | METAL | C3': 8, '6207 | CAUCHO | C3': 6, '6202 | METAL | C3': 22, '6202 | CAUCHO | C3': 12, '6303 | METAL | C3': 8, '6304 | CAUCHO | C3': 6, '6304 | METAL | C3': 6, '6201 | METAL | C3': 12, '6303 | CAUCHO | C3': 6, '6201 | CAUCHO | C3': 8, '6004 | METAL | C3': 6, '6305 | CAUCHO | C3': 2, '6302 | METAL | C3': 4, '6004 | CAUCHO | C3': 4, '6302 | CAUCHO | C3': 2, '6301 | CAUCHO | C3': 2, '6000 | CAUCHO | C3': 2, '6001 | CAUCHO | C3': 2, '608 | METAL | C3': 2, '608 | CAUCHO | C3': 2}
REFERENCIAS_DESPACHO = {'6314 | CAUCHO | C3': '6314-2RS1/C3GJN', '6312 | CAUCHO | C3': '6312-2RS1/C3GJN', '6311 | CAUCHO | C3': '6311-2RS1/C3GJN', '6313 | CAUCHO | C3': '6313-2RS1/C3GJN', '6313 | METAL | C3': '6313-2Z/C3GJN', '6309 | CAUCHO | C3': '6309-2RS1/C3GJN', '6206 | METAL | C3': '6206-2Z/C3GJN', '6310 | CAUCHO | C3': '6310-2RS1/C3GJN', '6309 | METAL | C3': '6309-2Z/C3GJN', '6312 | METAL | C3': '6312-2Z/C3GJN', '6308 | METAL | C3': '6308-2Z/C3GJN', '6308 | CAUCHO | C3': '6308-2RS1/C3GJN', '6306 | METAL | C3': '6306-2Z/C3GJN', '6208 | CAUCHO | C3': '6208-2RS1/C3GJN', '6306 | CAUCHO | C3': '6306-2RS1/C3GJN', '6206 | CAUCHO | C3': '6206-2RS1/C3GJN', '6208 | METAL | C3': '6208-2Z/C3GJN', '6311 | METAL | C3': '6311-2Z/C3GJN', '6205 | METAL | C3': '6205-2Z/C3GJN', '6209 | CAUCHO | C3': '6209-2RS1/C3GJN', '6205 | CAUCHO | C3': '6205-2RSH/C3GJN', '6307 | METAL | C3': '6307-2Z/C3GJN', '6207 | METAL | C3': '6207-2Z/C3GJN', '6212 | CAUCHO | C3': '6212-2RS1/C3GJN', '6212 | METAL | C3': '6212-2Z/C3GJN', '6210 | CAUCHO | C3': '6210-2RS1/C3GJN', '6211 | CAUCHO | C3': '6211-2RS1/C3GJN', '6204 | METAL | C3': '6204-2Z/C3GJN', '6203 | CAUCHO | C3': '6203-2RSH/C3GJN', '6209 | METAL | C3': '6209-2Z/C3GJN', '6204 | CAUCHO | C3': '6204-2RSH/C3GJN', '6210 | METAL | C3': '6210-2Z/C3GJN', '6307 | CAUCHO | C3': '6307-2RS1/C3GJN', '6203 | METAL | C3': '6203-2Z/C3GJN', '6305 | METAL | C3': '6305-2Z/C3GJN', '6207 | CAUCHO | C3': '6207-2RS1/C3GJN', '6202 | METAL | C3': '6202-2Z/C3GJN', '6202 | CAUCHO | C3': '6202-2RSH/C3GJN', '6303 | METAL | C3': '6303-2Z/C3GJN', '6304 | CAUCHO | C3': '6304-2RSH/C3GJN', '6304 | METAL | C3': '6304-2Z/C3GJN', '6201 | METAL | C3': '6201-2Z/C3GJN', '6303 | CAUCHO | C3': '6303-2RSH/C3GJN', '6201 | CAUCHO | C3': '6201-2RSH/C3GJN', '6004 | METAL | C3': '6004-2Z/C3GJN', '6305 | CAUCHO | C3': '6305-2RS1/C3GJN', '6302 | METAL | C3': '6302-2Z/C3GJN', '6004 | CAUCHO | C3': '6004-2RSH/C3GJN', '6302 | CAUCHO | C3': '6302-2RSH/C3GJN', '6301 | CAUCHO | C3': '6301-2RSH/C3GJN', '6000 | CAUCHO | C3': '6000-2RSH/C3GJN', '6001 | CAUCHO | C3': '6001-2RSH/C3GJN', '608 | METAL | C3': '608-2Z/C3GJN', '608 | CAUCHO | C3': '608-2RSH/C3GJN'}

pd.set_option("display.max_rows", 200)

## Carga, validación y cálculo

El consumo del periodo es `nivel acordado − unidades contadas + reposiciones`.
El acumulado usa hasta los últimos tres conteos disponibles. Después de tres conteos
sin consumo se recomienda retirar la referencia.

In [2]:
conteos = pd.read_csv(ARCHIVO_CONTEO)
obligatorias = {"referencia", "unidades_en_bodega", "fecha_conteo"}
faltantes = obligatorias - set(conteos.columns)
if faltantes:
    raise ValueError(f"Faltan columnas: {sorted(faltantes)}")

if "reposiciones" not in conteos:
    conteos["reposiciones"] = 0
    tratamiento_reposiciones = "ASUMIDAS_CERO"
else:
    tratamiento_reposiciones = "INFORMADAS_EN_ARCHIVO"

parsed = pd.DataFrame(
    conteos["referencia"].map(normalize_designation).tolist(),
    index=conteos.index,
).add_prefix("parser_")
conteos = pd.concat([conteos, parsed], axis=1)
if not conteos["parser_ok"].all():
    display(conteos.loc[~conteos["parser_ok"], ["referencia", "parser_error"]])
    raise ValueError("Hay referencias que el parser no reconoce.")

conteos["fecha_conteo"] = pd.to_datetime(conteos["fecha_conteo"], errors="raise")
conteos["unidades_en_bodega"] = pd.to_numeric(
    conteos["unidades_en_bodega"], errors="raise"
)
conteos["reposiciones"] = pd.to_numeric(conteos["reposiciones"], errors="raise")
if (conteos[["unidades_en_bodega", "reposiciones"]] < 0).any().any():
    raise ValueError("Conteos y reposiciones deben ser no negativos.")

niveles = pd.DataFrame({
    "llave_canonica": list(NIVELES_ACORDADOS),
    "nivel_acordado": list(NIVELES_ACORDADOS.values()),
})
niveles["referencia_skf_sugerida"] = niveles["llave_canonica"].map(
    REFERENCIAS_DESPACHO
)
conteos = conteos.merge(
    niveles,
    left_on="parser_llave_canonica",
    right_on="llave_canonica",
    how="left",
)
if conteos["nivel_acordado"].isna().any():
    display(conteos.loc[
        conteos["nivel_acordado"].isna(),
        ["referencia", "parser_llave_canonica"],
    ])
    raise ValueError("Hay referencias sin nivel acordado.")

conteos = conteos.sort_values(["llave_canonica", "fecha_conteo"])
conteos["consumo_periodo"] = (
    conteos["nivel_acordado"]
    - conteos["unidades_en_bodega"]
    + conteos["reposiciones"]
)
if conteos["consumo_periodo"].lt(0).any():
    raise ValueError("Hay consumo negativo; revise conteos o reposiciones.")

grupo = conteos.groupby("llave_canonica", group_keys=False)
conteos["numero_conteo"] = grupo.cumcount() + 1
conteos["consumo_ultimos_3"] = grupo["consumo_periodo"].transform(
    lambda s: s.rolling(CONTEOS_PARA_PROMEDIO, min_periods=1).sum()
)
conteos["consumo_mensual_promedio"] = (
    conteos["consumo_ultimos_3"]
    / conteos["numero_conteo"].clip(upper=CONTEOS_PARA_PROMEDIO)
)
conteos["meses_cobertura"] = np.where(
    conteos["consumo_mensual_promedio"] > 0,
    conteos["nivel_acordado"] / conteos["consumo_mensual_promedio"],
    np.nan,
)
conteos["nivel_recalculado"] = (
    conteos["consumo_mensual_promedio"] * MESES_COBERTURA
).map(ceil_even)

def ajuste(row):
    if row["numero_conteo"] >= 3 and row["consumo_ultimos_3"] == 0:
        return "RETIRAR"
    if row["nivel_recalculado"] > row["nivel_acordado"]:
        return "SUBIR"
    if row["nivel_recalculado"] < row["nivel_acordado"]:
        return "BAJAR"
    return "MANTENER"

conteos["ajuste_sugerido"] = conteos.apply(ajuste, axis=1)
ultimo = conteos.groupby("llave_canonica", as_index=False).tail(1).copy()
reporte = ultimo[[
    "referencia_skf_sugerida", "nivel_acordado", "unidades_en_bodega",
    "consumo_periodo", "consumo_ultimos_3", "meses_cobertura",
    "ajuste_sugerido",
]].rename(columns={
    "referencia_skf_sugerida": "Referencia",
    "nivel_acordado": "Nivel acordado",
    "unidades_en_bodega": "Unidades contadas",
    "consumo_periodo": "Consumo del periodo",
    "consumo_ultimos_3": "Consumo mensual acumulado (últimos 3 conteos)",
    "meses_cobertura": "Meses de cobertura del nivel actual",
    "ajuste_sugerido": "Ajuste sugerido",
})
print(f"Tratamiento de reposiciones: {tratamiento_reposiciones}")
display(reporte.style.format({
    "Nivel acordado": "{:,.0f}",
    "Unidades contadas": "{:,.0f}",
    "Consumo del periodo": "{:,.0f}",
    "Consumo mensual acumulado (últimos 3 conteos)": "{:,.1f}",
    "Meses de cobertura del nivel actual": "{:,.1f}",
}, na_rep="—"))

Tratamiento de reposiciones: ASUMIDAS_CERO


,Referencia,Nivel acordado,Unidades contadas,Consumo del periodo,Consumo mensual acumulado (últimos 3 conteos),Meses de cobertura del nivel actual,Ajuste sugerido
2,6206-2Z/C3GJN,38,20,18,40.0,2.9,BAJAR
5,6309-2RS1/C3GJN,12,6,6,12.0,3.0,BAJAR
